# Nishimoto et al. (2011) from CRCNS vim-2: color information

**Paper.** Nishimoto, Vu, Naselaris, Benjamini, Yu & Gallant (2011), Current Biology 21, 1641-1646, doi:10.1016/j.cub.2011.08.031. **Claims file.** `claims/nishimoto-2011-reconstructing.yaml`. **Dataset.** CRCNS vim-2, doi:10.6080/K00Z715X, version `unversioned` (CRCNS data sets are not versioned; docs/archives.md), streamed from the bucket `crcnsarchive` as in the first notebook, `reproduce_nishimoto_2011.ipynb`, which tests the other claims and describes the data.

This notebook tests one claim, **discussion-color-no-gain**. It is kept apart because it filters the movies three times (L*, A* and B*), which with the first notebook's analyses would pass the hour a notebook may take.

The claims file's reading of the color model: the directional model of fig2de-prediction-accuracy, with the same 6,555-filter bank applied as well to the A* and B* channels of CIE L*A*B* (19,665 filters, 78,660 regressors with the four delays), fit and scored the same way. Ridge regression is used, as in the first notebook (an assumption there), with the penalty of each voxel chosen on the held-out 10% of the training data (penalty reading a). Each channel's features are z-scored with their own training statistics, so the three channels enter the ridge penalty on the same scale (standard choice).

In [ ]:
import time, resource, collections, urllib.request
import numpy as np
from scipy import stats
import vim2

T0 = time.time()
SEED = 2011  # standard choice; the same held-out split as the first notebook
ALPHAS = np.logspace(0, 9, 19).astype(np.float32)
SUBJECTS = [1, 2, 3]
def mem_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6
t = time.time()
filters = vim2.pyramid().filters
S = vim2.stimulus_motion_energy(filters, ("L", "a", "b"))
ME = {c: (S["st"][c], S["sv"][c]) for c in ("L", "a", "b")}
stim_info = {k: S[k] for k in ("member", "member_bytes", "layout", "st_range", "sv_range", "st_L_range", "sv_L_range")}
del S
print(f"stimuli streamed, converted and filtered in three channels in {time.time() - t:.0f} s; peak memory {mem_gb():.1f} GB")

In [ ]:
md5 = urllib.request.urlopen(vim2.BASE + "checksums.md5", timeout=60).read().decode()
print("MD5 sums of the files read (checksums.md5):")
for line in md5.splitlines():
    if "Stimuli" in line or "VoxelResponses" in line:
        print(" ", line)
print("Stored arrays:", stim_info["layout"], "; pixel range (uint8)", stim_info["st_range"], stim_info["sv_range"])
print("L* range, training %.1f to %.1f, test %.1f to %.1f" % tuple(stim_info["st_L_range"] + stim_info["sv_L_range"]))
for c in ("L", "a", "b"):
    tr, te = ME[c]
    print(f"channel {c}: motion energy training {tr.shape}, test {te.shape}, range {tr.min():.2f} to {tr.max():.2f} (log units), "
          f"filters with no variance over training {(tr.std(0) == 0).sum()}")
print("Filters per channel:", len(filters))
ktr = vim2.keep_mask(7200, vim2.TRAIN_BLOCK)
kte = vim2.keep_mask(540, vim2.TEST_BLOCK)
print("Samples kept: training", ktr.sum(), "(paper 7,128), test", kte.sum(), "(paper 486)")

In [ ]:
held = vim2.holdout_blocks(int(ktr.sum()), np.random.default_rng([SEED, 0]))
cols = vim2.model_columns(filters, "directional")
KCH = {}
for c in ("L", "a", "b"):
    Xtr, Xte = vim2.build_features(*ME[c], cols)
    X = vim2.delayed(Xtr, vim2.DELAYS, 600)[ktr]
    Xe = vim2.delayed(Xte, vim2.DELAYS, 60)[kte]
    KCH[c] = (X @ X.T, Xe @ X.T)
    del X, Xe, Xtr, Xte
KER = {"luminance": KCH["L"], "color": tuple(sum(KCH[c][i] for c in ("L", "a", "b")) for i in (0, 1))}
del KCH, ME
print("regressors: luminance", len(cols) * 4, "; color", 3 * len(cols) * 4, f"; held-out samples {held.sum()}; peak memory {mem_gb():.1f} GB")
DATA = {}
for s in SUBJECTS:
    t = time.time()
    d = vim2.load_subject(s)
    good = ~(np.isnan(d["rt"]).any(0) | np.isnan(d["rva"]).any((0, 1)) | np.isnan(d["rv"]).any(0))
    rva = d["rva"][:, :, good]
    rva = (rva - rva.mean(1, keepdims=True)) / rva.std(1, keepdims=True)
    Yrep = rva[:, kte]
    DATA[s] = dict(Y=d["rt"][ktr][:, good], Yrep=Yrep, Yte=Yrep.mean(0), n_drop=int((~good).sum()), n=int(good.sum()))
    del d, rva
    print(f"subject {s}: {DATA[s]['n']} V1-V3B voxels kept, {DATA[s]['n_drop']} dropped for missing values; loaded in {time.time() - t:.0f} s")

In [ ]:
FIT = {s: {} for s in SUBJECTS}
idx_all = np.arange(int(ktr.sum()))
tr, ev = idx_all[~held], idx_all[held]
for m in ("luminance", "color"):
    t = time.time()
    K, Kte = KER[m]
    w, U = vim2.eig_split(K, tr)
    Kev = K[np.ix_(ev, tr)]
    for s in SUBJECTS:
        Y = DATA[s]["Y"]
        rh = vim2.ridge_path_corr(w, U, Kev, Y[tr], Y[ev], ALPHAS)
        FIT[s][m] = dict(ai=rh.argmax(0), held_acc=rh.max(0))
        if s == 1:
            C90 = vim2.ridge_dual(w, U, Y[tr], FIT[s][m]["ai"], ALPHAS)
            FIT[s][m]["Pheld"] = Kev @ C90
            FIT[s][m]["R90"] = Y[tr] - K[np.ix_(tr, tr)] @ C90
            del C90
    w, U = vim2.eig_split(K, idx_all)
    for s in SUBJECTS:
        Y, f = DATA[s]["Y"], FIT[s][m]
        C = vim2.ridge_dual(w, U, Y, f["ai"], ALPHAS)
        P = Kte @ C
        f["avg"] = vim2.corr_cols(P, DATA[s]["Yte"])
        f["rep"] = np.mean([vim2.corr_cols(P, DATA[s]["Yrep"][k]) for k in range(10)], 0)
        if s == 1:
            f["Pte"], f["R"] = P, Y - K @ C
        del C, P
    print(f"{m}: fitted in {time.time() - t:.0f} s; voxels per chosen penalty (S1) {np.bincount(FIT[1][m]['ai'], minlength=len(ALPHAS)).tolist()}")

## discussion-color-no-gain

> In preliminary work, encoding models with explicit color information did not predict or identify better than models using luminance alone.

Criterion: reproduced when the color model's mean accuracy is not higher by the paired Wilcoxon signed-rank test over voxels at p < 0.0001; the difference is reported either way. The identification count of each model (as fig3ab-identification-s1, model reading a, S1) is reported but does not decide the outcome. Readings of the test signal as in fig2de-prediction-accuracy: (a) the repeat mean, (b) the mean over single repeats; subjects (i) pooled or (ii) the mean of subject means.

In [ ]:
def pooled(m, rd):
    return np.concatenate([FIT[s][m][rd] for s in SUBJECTS])
for rd in ("avg", "rep"):
    L, C = pooled("luminance", rd), pooled("color", rd)
    p2 = stats.wilcoxon(C, L).pvalue
    p1 = stats.wilcoxon(C, L, alternative="greater").pvalue
    sm = {m: np.mean([FIT[s][m][rd].mean() for s in SUBJECTS]) for m in ("luminance", "color")}
    higher = C.mean() > L.mean() and p2 < 1e-4
    print(f"signal {rd}: (i) pooled over {len(L)} voxels, luminance {L.mean():.4f}, color {C.mean():.4f}, difference {C.mean() - L.mean():+.4f}; "
          f"(ii) subject means {sm['luminance']:.4f} and {sm['color']:.4f}; Wilcoxon two-sided p = {p2:.1e}, one-sided (color greater) p = {p1:.1e}; "
          f"color better than luminance in {np.mean(C > L):.2f} of voxels; color higher by the test: {higher}")
    for s in SUBJECTS:
        l, c = FIT[s]["luminance"][rd], FIT[s]["color"][rd]
        print(f"   S{s}: luminance {l.mean():.4f}, color {c.mean():.4f}, difference {c.mean() - l.mean():+.4f}, Wilcoxon two-sided p = {stats.wilcoxon(c, l).pvalue:.1e}")
D1 = DATA[1]
th = np.arange(len(held))[held]
tt = np.arange(540)[kte]
LAMS = np.round(np.logspace(-1, 2, 13), 3).tolist()
def within1(choice, times):
    return int(np.sum(np.abs(times[choice] - times) <= 1))
for m in ("luminance", "color"):
    f = FIT[1][m]
    sel = np.argsort(f["held_acc"])[::-1][:2000]
    cov90 = (f["R90"][:, sel].T @ f["R90"][:, sel]) / f["R90"].shape[0]
    hc = [within1(vim2.identify(D1["Y"][held][:, sel], f["Pheld"][:, sel], cov90, l), th) for l in LAMS]
    lam = LAMS[int(np.argmax(hc))]
    cov = (f["R"][:, sel].T @ f["R"][:, sel]) / f["R"].shape[0]
    ca = within1(vim2.identify(D1["Yte"][:, sel], f["Pte"][:, sel], cov, lam), tt)
    cr = [within1(vim2.identify(D1["Yrep"][k][:, sel], f["Pte"][:, sel], cov, lam), tt) for k in range(10)]
    print(f"identification, S1, {m}: Tikhonov penalty {lam} (held-out counts {hc} of {held.sum()}); "
          f"(i) repeat mean {ca} of 486; (ii) single repeats mean {np.mean(cr):.1f} (range {min(cr)} to {max(cr)})")
print(f"total run time {(time.time() - T0) / 60:.1f} min; peak memory {mem_gb():.1f} GB")

**Result: reproduced.** The color model (the 6,555-filter bank on L*, A* and B*) predicts worse than the luminance model, not better: pooled over 13,007 voxels the mean accuracy is 0.3639 against 0.3734 (difference -0.0095, repeat-averaged signal), lower in every subject (-0.0072, -0.0082, -0.0129), and better in only 38% of voxels; the one-sided Wilcoxon test of color greater gives p = 1.0, and single repeats give the same (-0.0047). In S1 identification the color model identifies 378 of 486 test volumes within one volume against 409 for luminance, which the criterion reports without a test. The outcome rests on the claims file's reading of the color model and on ridge regression, under which 78,660 regressors share one penalty per voxel.

## Summary

| Claim | Outcome |
|---|---|
| discussion-color-no-gain | reproduced (color 0.3639 against luminance 0.3734; identification 378 against 409 of 486) |

The other claims are in `reproduce_nishimoto_2011.ipynb`, with the data check and the differences from the paper's method. Here, the color model follows the claims file's reading (the paper does not describe its color models), and is fit by ridge regression with the penalty chosen on the held-out 10% of the training data, as in the first notebook.